# Chapter 6 - Low-Beta Interaction Region Insertion

## 6.1 Introduction

In this chapter, an interaction region (IR) with a low-beta interaction point (IP) is constructed by modifying the quadrupoles in the 6 o'clock straight section. The IR is divided into two parts: the upstream forward section before the IP, labeled `IPF`, and the downstream reverse section after the IP, labeled `IPR`.

At IP6, the target Twiss parameters are

$$
\beta_a^* = 0.6\ \mathrm{m}, \qquad
\beta_b^* = 0.06\ \mathrm{m}, \qquad
\alpha_a^* = \alpha_b^* = 0.
$$

The quadrupole strengths surrounding the IP will be adjusted to reach these targets while preserving the match to the periodic straight-section solution at the ends farthest from the IP. Because this region is particularly sensitive, the complete match is performed in three steps:

1. Match the forward line from the straight section to the IP.
2. Match backwards the reverse line from the straight section to the IP.
3. Connect the two lines and match the full IR to the periodic solution once more.

<div align="center"><img src="assets/chapter6_ir_overview.png" width="760"></div>

<p align="center"><em>Figure 1: A) Layout before modification. B) Layout with IR constructed. C) IR beta functions.</em></p>


## 6.2 Example: Constructing the Upstream and Downstream IR Lines

This example constructs only the geometry and element sequences of `IPF` and `IPR`. The final strengths of the new quadrupoles are not yet known, so their initial values use the straight-section FODO quadrupole strength

$$
K_{\mathrm{SS}} = 0.351957452649287\ \mathrm{m}^{-2}.
$$

The red circles mark drifts with non-standard lengths. `IP6` is a zero-length marker and has no effect on orbit or spin transport.

<div align="center"><img src="assets/chapter6_ir_dimensions.png" width="820"></div>

<p align="center"><em>Figure 2: Interaction region dimensions. Red circles mark drifts with non-standard lengths. IPF: Forward section before the IP. IPR: Reverse section after the IP.</em></p>


In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()


In [1]:
using SciBmad
using Printf

### Define the shared straight-section elements

The standard straight-section drifts are `D1`, `DB`, and `D2`. The special IR drifts and all IR quadrupoles are defined separately below so their lengths and strengths can later be varied independently.

In [2]:
species_ref = Species("electron")
E_ref = 18e9

K_ss = 0.351957452649287

element_specs = Dict(
    # Standard straight-section drifts
    :D1   => (:drift, 0.609, 0.0),
    :DB   => (:drift, 5.855, 0.0),
    :D2   => (:drift, 1.241, 0.0),

    # Forward (upstream) IR
    :QEF1 => (:quad,  0.5,  K_ss),
    :QEF2 => (:quad,  0.5, -K_ss),
    :DEF1 => (:drift, 20.46, 0.0),
    :QEF3 => (:quad,  1.6,  K_ss),
    :DEF2 => (:drift, 3.76, 0.0),
    :QEF4 => (:quad,  1.2, -K_ss),
    :DEF3 => (:drift, 5.8, 0.0),

    # Reverse (downstream) IR
    :DER3 => (:drift, 5.3, 0.0),
    :QER4 => (:quad,  1.8, -K_ss),
    :DER2 => (:drift, 0.5, 0.0),
    :QER3 => (:quad,  1.4,  K_ss),
    :DER1 => (:drift, 23.82, 0.0),
    :QER2 => (:quad,  0.5,  K_ss),
    :QER1 => (:quad,  0.5, -K_ss),

    # Interaction point
    :IP6  => (:marker, 0.0, 0.0),
)

function make_element(name::Symbol)
    kind, L, K1 = element_specs[name]
    kind == :drift  && return Drift(name=String(name), L=L)
    kind == :quad   && return Quadrupole(name=String(name), L=L, Kn1=K1)
    kind == :marker && return Marker(name=String(name))
    error("Unknown element kind: $(kind)")
end

function make_beamline(names)
    Beamline([make_element(name) for name in names]; species_ref=species_ref, E_ref=E_ref)
end

make_beamline (generic function with 1 method)

### Construct the forward (upstream) line `IPF`

`IPF` starts in the ordinary forward straight section and ends at `IP6`. The four adjustable quadrupoles for the later forward matching exercise are `QEF1`, `QEF2`, `QEF3`, and `QEF4`.

In [3]:
IPF_NAMES = [
    :QEF1, :D1, :DB, :D2,
    :QEF2, :D1, :DB, :D2,
    :DEF1, :QEF3, :DEF2, :QEF4, :DEF3, :IP6,
]

IPF = make_beamline(IPF_NAMES)
IPF

Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1       QEF1   Quadrupole   0
  2       D1     Drift        0.5
  3       DB     Drift        1.109
  4       D2     Drift        6.964
  5       QEF2   Quadrupole   8.205
  6       D1     Drift        8.705
  7       DB     Drift        9.314
  8       D2     Drift        15.169
  9       DEF1   Drift        16.41
  10      QEF3   Quadrupole   36.87
  11      DEF2   Drift        38.47
  12      QEF4   Quadrupole   42.23
  13      DEF3   Drift        43.43
  14      IP6    Marker       49.23


### Construct the reverse (downstream) line `IPR`

`IPR` begins at `IP6` and proceeds through the reverse straight section. The four adjustable quadrupoles for the later backward matching exercise are `QER1`, `QER2`, `QER3`, and `QER4`.

In [4]:
IPR_NAMES = [
    :IP6, :DER3, :QER4, :DER2, :QER3, :DER1,
    :QER2, :D2, :DB, :D1,
    :QER1, :D2, :DB, :D1,
]

IPR = make_beamline(IPR_NAMES)
IPR

Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1       IP6    Marker       0
  2       DER3   Drift        0.0
  3       QER4   Quadrupole   5.3
  4       DER2   Drift        7.1
  5       QER3   Quadrupole   7.6
  6       DER1   Drift        9.0
  7       QER2   Quadrupole   32.82
  8       D2     Drift        33.32
  9       DB     Drift        34.561
  10      D1     Drift        40.416
  11      QER1   Quadrupole   41.025
  12      D2     Drift        41.525
  13      DB     Drift        42.766
  14      D1     Drift        48.621


### Connect the two halves

The complete IR is `IPF` followed by `IPR`. Because both half-lines contain `IP6`, the second copy is removed when the full sequence is assembled.

In [5]:
IR_NAMES = vcat(IPF_NAMES, IPR_NAMES[2:end])
IR = make_beamline(IR_NAMES)

@assert count(==(:IP6), IR_NAMES) == 1
IR

Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1       QEF1   Quadrupole   0
  2       D1     Drift        0.5
  3       DB     Drift        1.109
  4       D2     Drift        6.964
  5       QEF2   Quadrupole   8.205
  6       D1     Drift        8.705
  7       DB     Drift        9.314
  8       D2     Drift        15.169
  9       DEF1   Drift        16.41
  10      QEF3   Quadrupole   36.87
  11      DEF2   Drift        38.47
  12      QEF4   Quadrupole   42.23
  13      DEF3   Drift        43.43
  14      IP6    Marker       49.23
  15      DER3   Drift        49.23
  16      QER4   Quadrupole   54.53
  17      DER2   Drift        56.33
  18      QER3   Quadrupole   56.83
  19      DER1   Drift        58.23
  ⋮       ⋮      ⋮            ⋮

                        8 rows omitted

### Inspect the constructed geometry

These checks verify the element order and physical lengths. At this stage, the quadrupole strengths are only initial guesses; no Twiss matching has yet been performed.

In [6]:
element_length(name::Symbol) = element_specs[name][2]
line_length(names) = sum(element_length(name) for name in names)

@printf("IPF length = %.3f m\n", line_length(IPF_NAMES))
@printf("IPR length = %.3f m\n", line_length(IPR_NAMES))
@printf("IR length  = %.3f m\n", line_length(IR_NAMES))

println("\nIPF sequence:\n", join(string.(IPF_NAMES), " -> "))
println("\nIPR sequence:\n", join(string.(IPR_NAMES), " -> "))

IPF length = 49.230 m
IPR length = 49.230 m
IR length  = 98.460 m

IPF sequence:
QEF1 -> D1 -> DB -> D2 -> QEF2 -> D1 -> DB -> D2 -> DEF1 -> QEF3 -> DEF2 -> QEF4 -> DEF3 -> IP6

IPR sequence:
IP6 -> DER3 -> QER4 -> DER2 -> QER3 -> DER1 -> QER2 -> D2 -> DB -> D1 -> QER1 -> D2 -> DB -> D1


## 6.3 Exercises

The example above deliberately stops after constructing the IR lines. The matching and optimization are the exercises, following Chapter 6 of the PDF.

### 1. Forward (upstream) interaction-region matching

Using `IPF`, vary the strengths of `QEF1`, `QEF2`, `QEF3`, and `QEF4` so that at `IP6`

$$
\beta_a = 0.6\ \mathrm{m}, \quad
\beta_b = 0.06\ \mathrm{m}, \quad
\alpha_a = \alpha_b = 0.
$$

Use a variable step size of $10^{-6}$ for the quadrupole $K_1$ values. Save the optimized strengths in `lattices/chapter_6/chapter6_IPF_solution.jl`.

### 2. Alpha and beta functions

Determine how the initial beta and alpha values must be changed when matching a reflected line. Use the reflection relations developed in the earlier chapters.

### 3. Reverse (downstream) interaction-region matching

Match `IPR` backwards: start from the end of the line and finish at `IP6`. Vary `QER1`, `QER2`, `QER3`, and `QER4` so that the same target Twiss parameters are reached at `IP6`. Save the optimized strengths in `lattices/chapter_6/chapter6_IPR_solution.jl`.

### 4. Load and rebuild the full IR

Load `lattices/chapter_6/chapter6_IPF_solution.jl` and `lattices/chapter_6/chapter6_IPR_solution.jl`, rebuild `IPF` and `IPR` using the optimized quadrupole strengths, and connect the two lines to form the full IR. Then vary the downstream quadrupoles `QER1`, `QER2`, `QER3`, and `QER4` once more so that the end of the full IR matches the periodic straight-section Twiss solution exactly.

### 5. Add the IR into the ring

Replace the 6 o'clock straight section in the Chapter 5 ring with the optimized IR. Following the PDF notation, modify the corresponding forward and reverse sextants so they contain

```text
SEXTANT5: 4*FODOSSF, SS_TO_ARCF, 20*FODOAF, ARC_TO_SSF, FODOSSF, IPF
SEXTANT7: IPR, FODOSSR, SS_TO_ARCR, 20*FODOAR, ARC_TO_SSR, 4*FODOSSR
```

### 6. Importance of variable step size

Repeat Exercise 1 using a quadrupole-strength step size of $10^{-4}$ instead of $10^{-6}$. Verify that the Levenberg-Marquardt optimization no longer converges to a usable solution, and explain why the finite-difference step size affects the calculated Jacobian and convergence.

In [ ]:
# Exercise starter values. This cell does not perform the matching.
target_ip = (beta_a=0.6, alpha_a=0.0, beta_b=0.06, alpha_b=0.0)
K_IPF_start = fill(K_ss, 4) .* [1, -1, 1, -1]
K_IPR_start = fill(K_ss, 4) .* [-1, 1, 1, -1]  # QER1, QER2, QER3, QER4
fd_step = 1e-6

@show target_ip K_IPF_start K_IPR_start fd_step